# Drug-likeness and ADMET prediction

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yboulaamane/comp_chem_colab/blob/main/admet_druglikeness.ipynb)

A molecule can bind its target and still fail as a drug if it is not absorbed, does not reach
the tissue, is cleared too fast or is toxic. This notebook screens a set of compounds in
three ways:

1. **Physicochemical rules** (Lipinski, Veber) and the QED drug-likeness score, from RDKit descriptors.
2. **Structural alerts** (PAINS, Brenk) that flag reactive or assay-interfering groups.
3. **Machine-learned ADMET endpoints** from [ADMET-AI](https://github.com/swansonk14/admet_ai):
   blood-brain barrier penetration, hERG, CYP inhibition, mutagenicity, solubility and more.

The example set is MAO-B inhibitors (a brain target, so getting across the blood-brain
barrier matters) and some natural coumarins. Swap in your own SMILES.

Runtime: CPU is fine.

## Setup

RDKit for the descriptors and alerts; ADMET-AI for the predictions (it installs its own
small models, no download at run time).

In [ ]:
%pip install -q rdkit admet-ai

In [ ]:
import numpy as np
import pandas as pd
from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors, Draw, QED
from rdkit.Chem.FilterCatalog import FilterCatalog, FilterCatalogParams

RDLogger.DisableLog("rdApp.*")
pd.set_option("display.max_columns", None)

In [ ]:
compounds = {
    "selegiline": "C#CCN(C)[C@H](C)Cc1ccccc1",
    "rasagiline": "C#CCN[C@@H]1CCc2ccccc21",
    "safinamide": "C[C@H](NCc1ccc(OCc2cccc(F)c2)cc1)C(N)=O",
    "coumarin": "O=c1ccc2ccccc2o1",
    "umbelliferone": "O=c1ccc2ccc(O)cc2o1",
    "scopoletin": "COc1cc2ccc(=O)oc2cc1O",
    "osthole": "COc1ccc2ccc(=O)oc2c1CC=C(C)C",
    "imperatorin": "CC(C)=CCOc1c2occc2cc2ccc(=O)oc12",
    "quercetin": "O=c1c(O)c(-c2ccc(O)c(O)c2)oc2cc(O)cc(O)c12",
}
frame = pd.DataFrame({"compound": list(compounds), "smiles": list(compounds.values())})
frame["mol"] = frame["smiles"].map(Chem.MolFromSmiles)
Draw.MolsToGridImage(list(frame["mol"]), legends=list(frame["compound"]), molsPerRow=3, subImgSize=(220, 160))

## 1. Physicochemical rules

Lipinski's rule of five and Veber's rules are rough filters for oral drug-likeness. They are
guides, not laws: many approved drugs break one, and CNS drugs tend to be smaller and less
polar than the limits. QED rolls several properties into one 0-to-1 desirability score.

In [ ]:
def properties(mol):
    return pd.Series({
        "MW": Descriptors.MolWt(mol),
        "cLogP": Descriptors.MolLogP(mol),
        "HBD": Descriptors.NumHDonors(mol),
        "HBA": Descriptors.NumHAcceptors(mol),
        "TPSA": Descriptors.TPSA(mol),
        "RotB": Descriptors.NumRotatableBonds(mol),
        "QED": QED.qed(mol),
    })


props = frame["mol"].apply(properties)


def lipinski_violations(row):
    return int(row.MW > 500) + int(row.cLogP > 5) + int(row.HBD > 5) + int(row.HBA > 10)


props["Lipinski_viol"] = props.apply(lipinski_violations, axis=1)
props["Veber_pass"] = (props.RotB <= 10) & (props.TPSA <= 140)
props.insert(0, "compound", frame["compound"])
props.round(2)

## 2. Structural alerts

PAINS flags substructures that often show up as false positives in assays; Brenk flags
reactive or otherwise undesirable groups. A hit is a reason to look closely, not an automatic
rejection. Quercetin, included on purpose, carries a catechol that both catalogs flag.

In [ ]:
params = FilterCatalogParams()
for name in ("PAINS", "BRENK"):
    params.AddCatalog(getattr(FilterCatalogParams.FilterCatalogs, name))
catalog = FilterCatalog(params)


def alerts(mol):
    hits = catalog.GetMatches(mol)
    labels = [f"{h.GetProp('FilterSet')}: {h.GetDescription()}" for h in hits]
    return pd.Series({"n_alerts": len(labels), "alerts": "; ".join(labels)})


frame.join(frame["mol"].apply(alerts))[["compound", "n_alerts", "alerts"]]

## 3. Machine-learned ADMET

ADMET-AI predicts about 40 endpoints with models trained on the Therapeutics Data Commons.
The first cell loads the models (a few seconds); the second predicts. Below is a readable
subset; `predictions` holds them all.

In [ ]:
from admet_ai import ADMETModel

model = ADMETModel()
predictions = model.predict(smiles=list(compounds.values()))
predictions.index = list(compounds)
print(f"{predictions.shape[1]} endpoints predicted")

A few that matter for a CNS drug. Classification columns are probabilities from 0 to 1;
`BBB_Martins` is the chance of crossing the blood-brain barrier (higher is better here),
while `hERG`, `AMES` and `DILI` are risk flags (lower is better). `Solubility_AqSolDB` is
log mol/L and `Caco2_Wang` is log permeability.

In [ ]:
endpoints = {
    "BBB_Martins": "BBB (higher=crosses)",
    "hERG": "hERG risk",
    "AMES": "Ames mutagenicity",
    "DILI": "Liver injury",
    "CYP2D6_Veith": "CYP2D6 inhibition",
    "CYP3A4_Veith": "CYP3A4 inhibition",
    "Solubility_AqSolDB": "Solubility (logS)",
    "Caco2_Wang": "Caco-2 perm.",
}
predictions[list(endpoints)].rename(columns=endpoints).round(2)

ADMET-AI also reports where each value falls among approved drugs (its DrugBank percentile),
which puts a raw number in context. The heatmap shows the classification endpoints; a redder
cell means a higher predicted probability.

In [ ]:
import matplotlib.pyplot as plt

risk = predictions[["BBB_Martins", "hERG", "AMES", "DILI", "CYP2D6_Veith", "CYP3A4_Veith",
                    "Pgp_Broccatelli", "Bioavailability_Ma"]]
fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(risk.values, cmap="RdYlGn_r", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(risk.shape[1]), risk.columns, rotation=45, ha="right")
ax.set_yticks(range(risk.shape[0]), risk.index)
for i in range(risk.shape[0]):
    for j in range(risk.shape[1]):
        ax.text(j, i, f"{risk.values[i, j]:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(im, label="predicted probability")
plt.tight_layout()
plt.show()

## Caveats

* These are statistical predictions from public data, not measurements. Use them to
  prioritise and to spot risks early, then confirm the important ones experimentally.
* The rules and alerts are heuristics. Approved drugs break them; a flagged group is a
  prompt to check, not a verdict.
* Predictions are least reliable for chemistry unlike the training data. Natural products
  with unusual scaffolds are a common blind spot.